In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from time import perf_counter

import pandas as pd
from astropy.table import Table
from astropy.io import fits

import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
anchors = Table.read(
    "/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_4000_sga152x152_complete.csv"
)

MAIN_TYPE_COL = "Main_type"

main_type_map = {
    20: "E",
    10: "S",
    0: "L",
    -5: "I"
}

# normalize ONCE
anchors["trainer_class"] = [
    main_type_map[m] for m in anchors[MAIN_TYPE_COL]
]

In [3]:
user_results = []

USERNAME = "Quillan"

total = 0
correct = 0
current_galaxy = None
start_time = None

In [4]:
def load_image(path):
    with fits.open(path) as hdul:
        data = hdul[0].data

    if data.ndim == 3 and data.shape[0] == 3:
        img = np.transpose(data, (1, 2, 0))
    else:
        img = data

    p1, p99 = np.percentile(img, [1, 99])
    if p99 > p1:
        img = np.clip(img, p1, p99)
        img = (img - p1) / (p99 - p1)

    return img


def cutout_path(row):
    tid = row["ref_id"]
    for folder in ["Elliptical", "Spiral", "Lenticular", "Irregular"]:
        pattern = f"/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/{folder}/{tid}_grz_152_*.fits"
        matches = glob.glob(pattern)
        if matches:
            return matches[0]
    raise FileNotFoundError(tid)

In [5]:
image_out = widgets.Output()
info_out = widgets.Output()

btn_E = widgets.Button(description="Elliptical", button_style="primary")
btn_L = widgets.Button(description="Lenticular")
btn_S = widgets.Button(description="Spiral")
btn_I = widgets.Button(description="Irregular")
btn_skip = widgets.Button(description="Skip")
btn_stop = widgets.Button(description="Stop", button_style="danger")

ui = widgets.HBox([btn_E, btn_L, btn_S, btn_I, btn_skip, btn_stop])

display(image_out, info_out, ui)

Output()

Output()

In [6]:
def new_galaxy():
    global current_galaxy, start_time

    idx = np.random.randint(len(anchors))
    current_galaxy = anchors[idx]
    start_time = perf_counter()

    img = load_image(cutout_path(current_galaxy))

    with image_out:
        image_out.clear_output(wait=True)
        plt.figure(figsize=(5,5))
        plt.imshow(img, origin="lower")
        plt.axis("off")
        plt.show()

    with info_out:
        info_out.clear_output(wait=True)
        print(f"Score: {correct}/{total} ({(correct/total*100) if total else 0:.1f}%)")
        print("Classify the galaxy:")

In [14]:
def handle_answer(choice):
    global total, correct, user_results

    if current_galaxy is None:
        return

    response_time = perf_counter() - start_time
    true = current_galaxy["trainer_class"]

    if choice == "skip":
        new_galaxy()
        return

    if choice == "stop":
        save()
        return

    is_correct = (choice == true)

    total += 1
    if is_correct:
        correct += 1

    user_results.append({
        "username": USERNAME,
        "target_id": int(current_galaxy["ref_id"]),
        "true_class": true,
        "user_class": choice,
        "correct": is_correct,
        "response_time": response_time
    })

    with info_out:
        info_out.clear_output(wait=True)
        print("✓ CORRECT!" if is_correct else "✗ INCORRECT")
        print(f"True: {true} | You: {choice}")
        print(f"Time: {response_time:.2f}s")
        print(f"Score: {correct}/{total}")
    
    # keep feedback visible longer
    import time
    time.sleep(0.5)
    
    new_galaxy()

In [8]:
btn_E.on_click(lambda x: handle_answer("E"))
btn_L.on_click(lambda x: handle_answer("L"))
btn_S.on_click(lambda x: handle_answer("S"))
btn_I.on_click(lambda x: handle_answer("I"))
btn_skip.on_click(lambda x: handle_answer("skip"))
btn_stop.on_click(lambda x: handle_answer("stop"))

In [9]:
def save():
    df = pd.DataFrame(user_results)
    path = f"/pscratch/sd/q/qshimp/VI_training/savefiles/{USERNAME}.csv"
    df.to_csv(path, index=False)
    print(f"Saved to {path}")

In [15]:
new_galaxy()